<a href="https://colab.research.google.com/github/hanxuanmeng/blood-cell-classification/blob/main/01_Data_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Exploring the Dataset

**Red Blood Cell Morphology Classification — Hanxuan**

In [ ]:
from pathlib import Path
import random, requests, zipfile, tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from google.colab import drive

drive.mount("/content/drive")
RANDOM_SEED=42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES=["Circular","Elongated","Other"]
CLASS_TO_INDEX={name:i for i,name in enumerate(CLASS_NAMES)}
IMAGE_SIZE=224
BATCH_SIZE=32
PROJECT_FOLDER=Path("/content/drive/MyDrive/Hanxuan_RBC_Classification")
DATA_FOLDER=PROJECT_FOLDER/"data"
RESULTS_FOLDER=PROJECT_FOLDER/"results"
MODELS_FOLDER=PROJECT_FOLDER/"models"
for folder in [DATA_FOLDER,RESULTS_FOLDER,MODELS_FOLDER]:
    folder.mkdir(parents=True,exist_ok=True)
print("PyTorch:",torch.__version__)
print("Device:",DEVICE)

## Obtain the official dataset
The code uses a cached copy when available; otherwise it reads the official Zenodo record.

In [ ]:
ZENODO_RECORD_ID="18299473"
ZENODO_API_URL=f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"
DOWNLOAD_FOLDER=DATA_FOLDER/"zenodo_download"
EXTRACTED_FOLDER=DATA_FOLDER/"erythrocytesIDB"
IMAGE_EXTENSIONS={".jpg",".jpeg",".png",".bmp",".tif",".tiff"}

def find_class_folders(root_folder):
    """Locate the three individual-cell morphology folders."""
    found={}
    for folder in Path(root_folder).rglob("*"):
        if folder.is_dir() and folder.name.lower() in {"circular","elongated","other"}:
            context="/".join(parent.name.lower() for parent in folder.parents)
            if "cell" in context or "individual" in context:
                found[folder.name.lower()]=folder
    return found

def get_dataset():
    """Use cached data or download files listed by the official Zenodo record."""
    if len(find_class_folders(EXTRACTED_FOLDER))==3:
        print("Using cached dataset.")
        return EXTRACTED_FOLDER
    DOWNLOAD_FOLDER.mkdir(parents=True,exist_ok=True)
    EXTRACTED_FOLDER.mkdir(parents=True,exist_ok=True)
    response=requests.get(ZENODO_API_URL,timeout=60)
    if response.status_code in (401,403):
        raise PermissionError("Dataset access is restricted. Follow DOI 10.5281/zenodo.18299473.")
    response.raise_for_status()
    files=response.json().get("files",[])
    if not files:
        raise RuntimeError("The Zenodo record currently exposes no downloadable files.")
    for info in files:
        name=info["key"]
        url=info.get("links",{}).get("content") or info.get("links",{}).get("self")
        if not url: continue
        local=DOWNLOAD_FOLDER/name
        if not local.exists():
            print("Downloading:",name)
            with requests.get(url,stream=True,timeout=120) as download:
                download.raise_for_status()
                with open(local,"wb") as output:
                    for chunk in download.iter_content(1024*1024):
                        if chunk: output.write(chunk)
        lower=name.lower()
        if lower.endswith(".zip"):
            with zipfile.ZipFile(local) as archive: archive.extractall(EXTRACTED_FOLDER)
        elif lower.endswith((".tar",".tar.gz",".tgz")):
            with tarfile.open(local) as archive: archive.extractall(EXTRACTED_FOLDER)
    if len(find_class_folders(EXTRACTED_FOLDER))!=3:
        raise RuntimeError("Circular, Elongated, and Other folders were not all detected.")
    return EXTRACTED_FOLDER

DATASET_FOLDER=get_dataset()

In [ ]:
from sklearn.model_selection import train_test_split

def build_image_table(dataset_folder):
    """Create one row for every labeled individual-cell image."""
    folders=find_class_folders(dataset_folder)
    records=[]
    for class_name in CLASS_NAMES:
        for image_path in folders[class_name.lower()].rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({"image_path":str(image_path),"label":class_name})
    return pd.DataFrame(records).drop_duplicates().reset_index(drop=True)

def split_dataset(image_table):
    """Create a fixed stratified 70/15/15 split."""
    training,remaining=train_test_split(image_table,test_size=0.30,random_state=RANDOM_SEED,stratify=image_table["label"])
    validation,test=train_test_split(remaining,test_size=0.50,random_state=RANDOM_SEED,stratify=remaining["label"])
    return training.reset_index(drop=True),validation.reset_index(drop=True),test.reset_index(drop=True)

class RedBloodCellDataset(Dataset):
    def __init__(self,image_table,transform):
        self.image_table=image_table.reset_index(drop=True)
        self.transform=transform
    def __len__(self):
        return len(self.image_table)
    def __getitem__(self,index):
        row=self.image_table.iloc[index]
        image=Image.open(row["image_path"]).convert("RGB")
        return self.transform(image),CLASS_TO_INDEX[row["label"]]

evaluation_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
training_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def create_data_loader(image_table,transform,shuffle):
    dataset=RedBloodCellDataset(image_table,transform)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=2,pin_memory=torch.cuda.is_available())

In [ ]:
image_table=build_image_table(DATASET_FOLDER)
print(f"Total individual-cell images: {len(image_table)}")
display(image_table.head())

## Class balance

In [ ]:
class_counts=image_table["label"].value_counts().reindex(CLASS_NAMES)
display(class_counts.rename("Number of images").to_frame())
class_counts.plot(kind="bar")
plt.title("Images in Each Morphology Class"); plt.xlabel("Class"); plt.ylabel("Images"); plt.xticks(rotation=0); plt.show()

## Representative examples

In [ ]:
figure,axes=plt.subplots(3,4,figsize=(12,8))
for row_index,class_name in enumerate(CLASS_NAMES):
    examples=image_table[image_table["label"]==class_name].sample(4,random_state=RANDOM_SEED)
    for column_index,(_,example) in enumerate(examples.iterrows()):
        image=Image.open(example["image_path"]).convert("RGB")
        axes[row_index,column_index].imshow(image)
        axes[row_index,column_index].set_title(class_name)
        axes[row_index,column_index].axis("off")
plt.suptitle("Representative Red Blood Cell Images"); plt.tight_layout(); plt.show()